# 用 Sciverse 查找论文全文证据

> 从检索片段出发，定位并读取原文完整段落作为可引用证据

**场景**: Agent 通过 agentic-search 找到了相关片段，但需要更完整的上下文来确认论点或生成精确引用。

**使用接口**: agentic-search, content

**预估调用量**: ~3–8 次 API 调用

---


## Step 1: 环境准备

安装依赖并配置环境变量


In [ ]:
!pip install httpx
import os
os.environ["SCIVERSE_API_TOKEN"] = "sv-your-token-here"  # 替换为你的真实值


## Step 2: 读取完整上下文

调用 content 接口，以 offset 为起点读取原文。响应字段为 text（非 content）


In [ ]:
import os
import asyncio
import httpx

BASE = "https://api.sciverse.space"
TOKEN = os.environ["SCIVERSE_API_TOKEN"]
HEADERS = {"Authorization": f"Bearer {TOKEN}"}

async def get_fulltext(doc_id: str, offset: int = 0, limit: int = 2000):
    """读取文档原文。返回 {text, next_offset, more}"""
    async with httpx.AsyncClient(timeout=30) as client:
        resp = await client.get(
            f"{BASE}/content",
            headers=HEADERS,
            params={"doc_id": doc_id, "offset": offset, "limit": limit}
        )
        resp.raise_for_status()
        return resp.json()

async def main():
    # 假设 agentic-search 返回了这个 hit
    hit = {"doc_id": "af2_nature_2021", "offset": 12480, "score": 0.94}

    # 向前偏移 300 字符以获取前文语境
    start = max(0, hit["offset"] - 300)
    result = await get_fulltext(hit["doc_id"], offset=start, limit=2000)

    print(f"Text length: {len(result['text'])} chars")
    print(f"Has more: {result['more']}")
    if result.get("next_offset"):
        print(f"Next offset: {result['next_offset']}")
    print(f"\
Content preview:\
{result['text'][:300]}...")
    return result

result = asyncio.run(main())


## Step 3: 迭代读取全文（可选）

如果需要更多上下文，使用 next_offset 循环读取


In [ ]:
async def read_full_document(doc_id: str, max_chars: int = 16000):
    """循环读取直到全文或达到字符上限"""
    full_text = ""
    offset = 0
    while len(full_text) < max_chars:
        result = await get_fulltext(doc_id, offset=offset, limit=4000)
        full_text += result["text"]
        if not result.get("more"):
            break
        offset = result["next_offset"]
    return full_text

async def main():
    text = await read_full_document("af2_nature_2021", max_chars=16000)
    print(f"Total document length: {len(text)} chars")

asyncio.run(main())


## 注意事项

- content 接口响应字段是 text（不是 content），请注意区分
- offset 是 Unicode 码点数，不是字节数
- 默认 limit=700 字符，建议传入 2000–4000 以减少调用次数
- 部分文档可能无全文（返回 404），需做异常处理
- 建议向前偏移 300–500 字符读取，以获取片段的前文语境


---

## 下一步

- [申请 API Token](https://sciverse.opendatalab.com/docs#auth)
- [查看完整 API 文档](https://sciverse.opendatalab.com/docs#sciverse/api)
- [更多 Cookbook 案例](https://sciverse.opendatalab.com/docs#cookbook)
